In [1]:
# ===================== 配置区（按需修改） =====================
IMGSZ   = 1024        # 推理分辨率（与训练一致）
CONF    = 0.33        # 置信度阈值（验证集 F1 最优折中点，可调）
IOU     = 0.7         # NMS IoU 阈值
MAX_DET = 300         # 单图最大检测数
SEED    = 42
DEVICE  = 0           # GPU 索引；无 GPU 改为 "cpu"

# 权重：把 <your-dataset> 换成你上传的数据集名
WEIGHTS_PATH = "/kaggle/input/datasets/beijietransient/gwhd-best/best.pt"
# 竞赛数据（Kaggle 标准路径）
COMP_DIR     = "/kaggle/input/competitions/global-wheat-detection"
TEST_DIR     = f"{COMP_DIR}/test"
# ==============================================================

import os, glob, random
import numpy as np
import pandas as pd
random.seed(SEED); np.random.seed(SEED)
print("配置完成：", dict(IMGSZ=IMGSZ, CONF=CONF, IOU=IOU, WEIGHTS=WEIGHTS_PATH))

配置完成： {'IMGSZ': 1024, 'CONF': 0.33, 'IOU': 0.7, 'WEIGHTS': '/kaggle/input/datasets/beijietransient/gwhd-best/best.pt'}


In [2]:
import glob, subprocess, sys

# 竞赛提交环境禁用网络：若镜像未预装 ultralytics，则从挂载的 gwhd-wheels 数据集离线安装
try:
    import ultralytics
except ImportError:
    whls = sorted(glob.glob("/kaggle/input/**/*.whl", recursive=True))
    assert whls, "未找到离线 wheel 包，请确认已挂载 gwhd-wheels 数据集（含 *.whl）。"
    wheel_dir = os.path.dirname(whls[0])
    print("离线安装 ultralytics 从:", wheel_dir)
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "--no-index", "--find-links", wheel_dir, "ultralytics"])
    import ultralytics  # 安装后重新导入

import torch
print("torch:", torch.__version__, "| CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("ultralytics:", ultralytics.__version__)
from ultralytics import YOLO
print("环境就绪")

离线安装 ultralytics 从: /kaggle/input/datasets/beijietransient/gwhd-wheels/gwhd-wheels
Looking in links: /kaggle/input/datasets/beijietransient/gwhd-wheels/gwhd-wheels
Processing /kaggle/input/datasets/beijietransient/gwhd-wheels/gwhd-wheels/ultralytics-8.4.137-py3-none-any.whl
Processing /kaggle/input/datasets/beijietransient/gwhd-wheels/gwhd-wheels/ultralytics_thop-2.1.6-py3-none-any.whl (from ultralytics)
Processing /kaggle/input/datasets/beijietransient/gwhd-wheels/gwhd-wheels/ultralytics_platform-0.1.21-py3-none-any.whl (from ultralytics)
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
torch: 2.10.0+cu128 | CUDA 可用: True
GPU: Tesla T4
ultralytics: 8.4.137
环境就绪


In [3]:
assert os.path.exists(WEIGHTS_PATH), f"找不到权重: {WEIGHTS_PATH}，请检查是否已挂载数据集"
model = YOLO(WEIGHTS_PATH)
print("模型加载成功:", WEIGHTS_PATH)

模型加载成功: /kaggle/input/datasets/beijietransient/gwhd-best/best.pt


In [4]:
def predict_image(path):
    res = model.predict(source=path, imgsz=IMGSZ, conf=CONF, iou=IOU,
                        max_det=MAX_DET, device=DEVICE, verbose=False)[0]
    boxes = res.boxes
    if boxes is None or len(boxes) == 0:
        return []
    xyxy  = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()
    order = np.argsort(-confs)          # 置信度降序
    out = []
    for i in order:
        x1, y1, x2, y2 = (float(v) for v in xyxy[i])
        out.append((float(confs[i]), x1, y1, x2 - x1, y2 - y1))
    return out

# 自动定位测试图像：标准路径 -> 任意挂载下的 test/ -> 任意挂载下所有 .jpg
def find_test_images():
    std = sorted(glob.glob(f"{TEST_DIR}/*.jpg"))
    if std:
        return std, TEST_DIR
    cands = sorted(glob.glob("/kaggle/input/**/test/*.jpg", recursive=True))
    if cands:
        return cands, "自动发现: **/test/*.jpg"
    cands = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True))
    return cands, "自动发现: **/*.jpg"

print("已挂载的输入目录:", os.listdir("/kaggle/input/"))
for _d in sorted(os.listdir("/kaggle/input/")):
    _p = f"/kaggle/input/{_d}"
    if os.path.isdir(_p):
        try:
            print(f"  [{_d}] -> {os.listdir(_p)[:12]}")
        except Exception:
            pass

test_images, _src = find_test_images()
assert test_images, (f"未找到任何测试图片，请确认已挂载竞赛数据（global-wheat-detection）。"
                     f"当前挂载: {os.listdir('/kaggle/input/')}")
print(f"测试图片来源: {_src}")
print(f"测试图像数量: {len(test_images)}")

rows, total = [], 0
for idx, path in enumerate(test_images, 1):
    image_id = os.path.splitext(os.path.basename(path))[0]
    preds = predict_image(path)
    total += len(preds)
    ps = " ".join(f"{c:.6f} {x:.2f} {y:.2f} {w:.2f} {h:.2f}" for c, x, y, w, h in preds)
    rows.append({"image_id": image_id, "PredictionString": ps})
    if idx % 50 == 0 or idx == len(test_images):
        print(f"  已处理 {idx}/{len(test_images)}，本图 {len(preds)} 框")

print(f"全部完成，共 {total} 个预测框")

已挂载的输入目录: ['competitions', 'datasets']
  [competitions] -> ['global-wheat-detection']
  [datasets] -> ['beijietransient']
测试图片来源: /kaggle/input/competitions/global-wheat-detection/test
测试图像数量: 10
  已处理 10/10，本图 20 框
全部完成，共 213 个预测框


In [5]:
sub = pd.DataFrame(rows, columns=["image_id", "PredictionString"])
# 显式保存到 Kaggle 输出目录 /kaggle/working（该目录文件 = 版本 Output）
out_csv = os.path.join(os.getcwd(), "submission.csv")
sub.to_csv(out_csv, index=False)
print("已保存提交文件:", out_csv)
print("文件是否存在:", os.path.exists(out_csv), "| 大小:", os.path.getsize(out_csv), "字节")
assert os.path.exists(out_csv), "提交文件未生成！"
print("✅ 文件已就绪，可点右上角 Save Version，成功后即可 Submit to Competition")

# ---- 格式校验 ----
import csv
ok = True
with open("submission.csv", encoding="utf-8") as f:
    rd = list(csv.reader(f))
assert rd[0] == ["image_id", "PredictionString"], "表头不正确"
for r in rd[1:]:
    toks = r[1].split() if len(r) > 1 and r[1] else []
    if len(toks) % 5 != 0:
        ok = False; print("格式错误:", r[0])
    for i in range(0, len(toks), 5):
        c, x, y, w, h = map(float, toks[i:i+5])
        if not (0 <= c <= 1 and 0 <= x <= 1024 and 0 <= y <= 1024 and w > 0 and h > 0):
            ok = False; print("数值越界:", r[0])
print("格式校验:", "通过 ✅" if ok else "失败 ❌")
print(f"共 {len(rd)-1} 张图像，总框数 {total}")
print(sub.head())


已保存提交文件: /kaggle/working/submission.csv
文件是否存在: True | 大小: 7717 字节
✅ 文件已就绪，可点右上角 Save Version，成功后即可 Submit to Competition
格式校验: 通过 ✅
共 10 张图像，总框数 213
    image_id                                   PredictionString
0  2fd875eaa  0.806534 0.00 907.65 108.16 88.18 0.799770 942...
1  348a992bb  0.787595 730.47 213.77 149.08 99.71 0.783341 0...
2  51b3e36ab  0.783934 1.44 810.74 107.06 109.19 0.781213 0....
3  51f1be19e  0.685239 500.96 463.72 194.92 113.45 0.683286 ...
4  53f253011  0.783674 10.75 26.17 153.57 116.38 0.778413 78...
